# Aftershock on Google Colab

Runs a mutation panel on a Colab GPU and keeps everything on your Google Drive.
The default panel is `configs/znf_panel_20ns.json`: 4 scenarios × 2 replicas = 16 simulations of 20 ns
of the zinc finger 1ZNF, followed by the WT-vs-mutant analysis and the WT-vs-WT null control.

**Before you start**

1. **GPU**: *Runtime → Change runtime type → A100 GPU* (the notebook asks for it; any NVIDIA GPU works, only slower).
2. **Secrets** (key icon in the left bar; switch on *Notebook access* for each):
   - `KEY_MODELLER`: your MODELLER licence key.
   - `GITHUB_TOKEN`: only while the repository is private. A token that can read `juvenalyosa/aftershock`
     (GitHub → Settings → Developer settings → Fine-grained tokens → *Contents: Read-only* on that repository).

   If a secret is missing, the cell asks for it instead.
3. Run the cells from top to bottom. Step 4 is a short test of the whole chain (a few minutes); step 5 starts the real run.

**If the session disconnects** nothing is lost: results and checkpoints are on Drive.
Open the notebook again, run steps 1, 2, 3, 5 and 6 (skip 4), and every simulation continues from its last checkpoint.

## 1. GPU and Google Drive

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0, "This runtime has no GPU. Runtime > Change runtime type > A100 GPU, then run this cell again."
print("GPU:", gpu.stdout.strip())

from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/aftershock_runs"   # results, checkpoints and logs
PANEL = "znf_panel_20ns"                              # name of a file in configs/ (without .json)
PANEL_DIR = f"{RUN_ROOT}/{PANEL}"
LOG = f"{RUN_ROOT}/{PANEL}.log"
print("Results will be in:", PANEL_DIR)

## 2. Get the code

In [ ]:
import base64, os, subprocess, sys

REPO = "juvenalyosa/aftershock"
CODE = "/content/aftershock"


def secret(name, description):
    """A Colab secret, or a prompt when it is not defined (the value is never printed)."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    from getpass import getpass
    return getpass(f"{description} ({name}): ").strip()


def git(*args, token=None):
    # The token travels in a header of this one command; it is not written to the clone.
    command = ["git"]
    if token:
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        command += ["-c", f"http.extraheader=AUTHORIZATION: basic {auth}"]
    return subprocess.run(command + list(args), capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})


def fetch(token=None):
    if os.path.isdir(f"{CODE}/.git"):
        return git("-C", CODE, "pull", "--ff-only", token=token)
    return git("clone", f"https://github.com/{REPO}.git", CODE, token=token)


result = fetch()
if result.returncode != 0:   # private repository: a token is needed
    result = fetch(secret("GITHUB_TOKEN", "GitHub token that can read the repository"))
assert result.returncode == 0, "Could not get the code:\n" + result.stderr
print("Code:", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

sys.path.insert(0, f"{CODE}/scripts")
import colab_tools as ct

## 3. Install OpenMM (CUDA) and MODELLER
A few minutes. Safe to run again.

In [ ]:
code = ct.run_stream(
    ["bash", f"{CODE}/scripts/colab_install.sh"],
    env={"KEY_MODELLER": secret("KEY_MODELLER", "MODELLER licence key"), "PERT_PYTHON": sys.executable},
)
assert code == 0, "Installation failed; the reason is in the lines above."

## 4. Short test of the whole chain

Two scenarios, one replica, 40 ps each: model building, GPU simulation, analysis, null control and reports,
on the local disk of the runtime. If this cell ends with `Test passed`, the long run will not fail for a
reason that could have been found here. Its numbers mean nothing (the simulations are far too short).

In [ ]:
SMOKE_DIR = "/content/aftershock_smoke"
smoke_config = ct.derive_config(
    f"{CODE}/configs/{PANEL}.json", "/content/aftershock_smoke.json",
    output_dir=SMOKE_DIR, scenarios="wt_model,snp_k2r", replicas=1, num_models=2,
    equil_steps=2000, prod_steps=20000, report_interval=100, checkpoint_interval_steps=10000, te_surrogates=16,
)
code = ct.run_stream(
    ["bash", f"{CODE}/scripts/run_panel_workflow.sh", smoke_config],
    env={"PERT_PYTHON": sys.executable, "CONTROL_SCENARIO": "wt_model"},
)
assert code == 0, "The test failed; the reason is in the lines above. Do not start the long run yet."

platforms = set(ct.platforms_used(SMOKE_DIR).values())
print("\nOpenMM platform used:", ", ".join(sorted(platforms)))
if platforms != {"CUDA"}:
    print("WARNING: the simulations did not run on CUDA. The long run would work, but much more slowly.")
print("Test passed.")

## 5. Start the real run

Starts the workflow in the background and returns at once. Running this cell again while the workflow is
running does nothing; running it after a disconnection resumes the panel.

To run a shorter or longer panel, add for example `prod_steps=2500000` (5 ns) to `derive_config` below.
Finished simulations are extended, not repeated, when `prod_steps` grows.

In [ ]:
config = ct.derive_config(
    f"{CODE}/configs/{PANEL}.json", f"{RUN_ROOT}/{PANEL}.json",
    output_dir=PANEL_DIR,
    checkpoint_interval_steps=1000000,   # a checkpoint every 2 ns of simulation
)
ct.launch(config, LOG, control_scenario="wt_model")

## 6. Watch it

Prints the log as it grows and, every ten minutes, how far the simulations are and an estimate of the time left.
Keep this cell running and the tab open: Colab closes sessions it believes idle.
Stopping this cell does **not** stop the run; run the cell again to keep watching.

In [ ]:
exit_code = ct.follow(LOG, output_dir=PANEL_DIR, config_path=config)

## 7. Results

The question the panel answers is in the first table: for each scenario, is the WT-vs-mutant difference
larger than the difference between two WT simulations? (`evidence`: none / suggestive / supported.)
The per-replica numbers behind it are in `null_control/null_calibrated_metrics.csv`.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

assert ct.exit_code(LOG) == 0, "The workflow has not finished (or failed): see step 6."

summary = json.load(open(f"{PANEL_DIR}/null_control/null_control_summary.json"))
verdicts = pd.DataFrame([
    {
        "scenario": name,
        "replicas": entry["replicas"],
        "evidence": entry["evidence"],
        "metrics beyond the null": ", ".join(entry["metrics_beyond_null"]) or "-",
        "note": entry["evidence_note"],
    }
    for name, entry in summary["scenarios"].items()
])
print(f"Null control: {summary['null_pairs']} WT-vs-WT pairs from {summary['wt_trajectories']} WT trajectories.")
for note in summary.get("notes", []):
    print(" -", note)
with pd.option_context("display.max_colwidth", None):
    display(verdicts)
display(Image(f"{PANEL_DIR}/null_control/null_control.png"))
display(Markdown(open(f"{PANEL_DIR}/allostery_evidence_report.md").read()))

Download tables, figures and reports (without the trajectories, which stay on Drive):

In [ ]:
from google.colab import files
files.download(ct.zip_light_results(PANEL_DIR, f"/content/{PANEL}_results.zip"))

## Stop the run
Only if you want to. Step 5 resumes it later from the last checkpoints.

In [ ]:
# ct.stop(LOG)